In [ ]:
# Análisis de precios de combustibles - DataFuel
# Dataset: http://datos.energia.gob.ar/dataset/1c181390-5045-475e-94dc-410429be4b17/archivo/80ac25de-a44a-4445-9215-090cf55cfda5

In [ ]:
# Secciones: 
# 4. Limpieza de datos
# 5. Evaluación de calidad de datos
# 6. Análisis Exploratorio (EDA)
# 7. Predicción de precios (ML con Scikit-Learn)

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Configuración
plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option('display.max_columns', None)

In [ ]:
# Carga del dataset
url = 'http://datos.energia.gob.ar/dataset/1c181390-5045-475e-94dc-410429be4b17/archivo/80ac25de-a44a-4445-9215-090cf55cfda5'
df = pd.read_csv(url, sep=';', encoding='utf-8')

In [ ]:
# Limpieza
df = df[df['region'].str.contains("BUENOS AIRES", na=False)]
df = df.dropna(subset=['region'])
df = df.drop_duplicates()
columnas_a_eliminar = ['idhorario', 'tipohorario', 'indice_tiempo', 'geojson']
df = df.drop(columns=[c for c in columnas_a_eliminar if c in df.columns])

In [ ]:
# Normalización de texto
class NormalizeText(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.str.lower().str.strip()

In [ ]:
pipeline = Pipeline([('normalize', NormalizeText())])
df['razon_social'] = pipeline.fit_transform(df['razon_social'])
df['producto'] = pipeline.fit_transform(df['producto'])

In [ ]:
# Conversión de tipos
df['fecha'] = pd.to_datetime(df['fecha'])
df['precio'] = pd.to_numeric(df['precio'], errors='coerce')
df = df.dropna(subset=['precio'])
df['region'] = df['region'].astype('category')

In [ ]:
# Evaluación de calidad
missing_percent = df.isnull().mean() * 100
print("Porcentaje de valores faltantes:\n", missing_percent)
duplicates = df.duplicated().sum()
print("\nCantidad de duplicados:", duplicates)
print("\nCategorías únicas en producto:\n", df['producto'].unique())

In [ ]:
# EDA
df['año'] = df['fecha'].dt.year
df['mes'] = df['fecha'].dt.month

In [ ]:
plt.figure(figsize=(12,6))
sns.lineplot(data=df, x='fecha', y='precio', hue='producto')
plt.title("Evolución temporal del precio por tipo de combustible")
plt.xlabel("Fecha")
plt.ylabel("Precio ($)")
plt.legend(title='Producto')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='producto', y='precio', estimator=np.mean, ci=None)
plt.title("Precio promedio por tipo de combustible")
plt.ylabel("Precio promedio ($)")
plt.xlabel("Producto")
plt.xticks(rotation=45)
plt.show()

In [ ]:
std_mensual = df.groupby(['año', 'mes', 'producto'])['precio'].std().reset_index()
plt.figure(figsize=(12,6))
sns.lineplot(data=std_mensual, x='mes', y='precio', hue='producto', style='año', markers=True)
plt.title("Volatilidad mensual del precio por tipo de combustible")
plt.ylabel("Desvío estándar del precio")
plt.show()

In [ ]:
print("\n--- INSIGHTS ---")
print("1. La Nafta súper tiene una tendencia de precios al alza con baja volatilidad.")
print("2. GNC muestra mayor variabilidad de precios entre meses.")
print("3. Diesel premium suele tener los precios más elevados en promedio.")

In [ ]:
# Exportar dataset limpio
df.to_csv("combustibles_limpio.csv", index=False)
print("Archivo exportado: combustibles_limpio.csv")

In [ ]:
# ML: predicción de precios
df_modelo = df.copy()
df_modelo['fecha_ordinal'] = df_modelo['fecha'].map(pd.Timestamp.toordinal)
X = df_modelo[['fecha_ordinal', 'producto']]
y = df_modelo['precio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
preprocesador = ColumnTransformer(
    transformers=[('producto_encoder', OneHotEncoder(handle_unknown='ignore'), ['producto'])],
    remainder='passthrough'
)

In [ ]:
modelo = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('regresion', LinearRegression())
])

In [ ]:
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

In [ ]:
# Predicción futura
fecha_futura = pd.to_datetime("2026-01-01").toordinal()
nuevo = pd.DataFrame({'fecha_ordinal': [fecha_futura], 'producto': ['nafta súper']})
prediccion_futura = modelo.predict(nuevo)
print(f"Precio estimado para Nafta Súper en enero 2026: ${prediccion_futura[0]:.2f}")